### 1. 申請自己的 API 金鑰

Mistral AI 金鑰 (可免費使用)

請至 https://console.mistral.ai/ 註冊，並選擇 plan (可以選免費的), 接著就可以申請 Mistral AI 的金鑰。請把這個鑰存在左方鑰匙的部份, 以 "Mistral" 的名稱存起來。

#### 讀入你的金鑰

需要多打以下程式碼，才能讀到Mistral的API Key
```
!pip install mistralai
```

In [ ]:
!pip install aisuite[all]
!pip install mistralai

In [ ]:
import aisuite as ai

In [ ]:
import os
from google.colab import userdata

`os.environ`需要是`MISTRAL_API_KEY`而不是`MISTRAL`

In [ ]:
#【使用 Mistral】
api_key = userdata.get('Mistral')
os.environ['MISTRAL_API_KEY']=api_key
provider = "mistral"
model = "ministral-8b-latest"

### 2. 打造唬爛產生器

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

基本上過去的對話紀錄長這個樣子。

    messages = [{"role":"system", "content":"ChatGPT的「人設」"},
            {"role": "user", "content": "使用者說"},
            {"role": "assistant", "content": "ChatGPT回應"},
            ：
            ：
            {"role": "user", "content": prompt (最後說的)}]

In [ ]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider="mistral",
          model="ministral-8b-latest"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content

請先為你的對話機器人做角色設定。

In [ ]:
system="""
你是一個「唬爛產生器」，
專門用台灣常用的繁體中文生成聽起來很有哲理、實際上內容空洞的文字。
語氣文青、假掰、充滿抽象詞彙（如存在、命運、本質）。
可引用中西方名言，如「孔子說…」「尼采說…」「莎士比亞說…」，
讓句子顯得深奧。
內容要流暢、有氣勢，
但不要給出具體事實或結論，
只要讓人覺得你好像講了什麼就行。
"""

試用一下 (ministral-8b-latest)

在prompt輸入主題

In [ ]:
prompt = "先有雞還是先有蛋?"
print(reply(system=system, prompt=prompt))

### 3. 用 Gradio 打造 Web App

**需先執行上面程式碼，以下程式才能執行**

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def lucky_post(prompt):
    response = reply(system=system,
                     prompt=prompt,
                     provider = provider,
                     model = model
                    )
    return response

In [ ]:
with gr.Blocks(title="唬爛產生器") as demo:
    gr.Markdown("### 唬爛產生器 🤪")
    gr.Markdown("請輸入一個主題，讓我幫你產生無意義的廢文，超棒的作文湊字數方式！")

    with gr.Row():
        user_input = gr.Textbox(label="主題是…", placeholder="例如：先有雞還是先有蛋?")

    submit_btn = gr.Button("一鍵生成文章!")
    output = gr.Textbox(label="🤡 看似有意義的文章")

    submit_btn.click(fn=lucky_post, inputs=user_input, outputs=output)

In [ ]:
demo.launch(share=True, debug=True)